In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install datasets tokenizers lightgbm numpy tqdm
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
CUDA: True NVIDIA A100-SXM4-40GB


In [14]:
import os, json, textwrap

ROOT = "/content/regmix"
for d in ["", "/data_raw/indic", "/data_bin", "/runs", "/src"]:
    os.makedirs(ROOT + d, exist_ok=True)

config = {
    "domains": ["web", "code", "maths", "indic", "papers"],
    "hf_sources": {
        "web":    {"id": "OptimalScale/ClimbMix",        "config": None,        "split": "train", "text_key": "text"},
        "code":   {"id": "bigcode/the-stack-dedup",       "config": "data/python","split": "train", "text_key": "content"},
        "maths":  {"id": "open-web-math/open-web-math",    "config": None,        "split": "train", "text_key": "text"},
        "papers": {"id": "togethercomputer/RedPajama-Data-1T", "config": "arxiv","split": "train", "text_key": "text"}
    },
    "local_sources": {"indic": "/content/regmix/data_raw/indic"},
    "tokenizer": "bigscience/bloom",   # multilingual — handles Indic far better than GPT-2
    "tokens_per_domain": 60_000_000,   # ~300M total pool to sample 50 proxies from
    "val_tokens_per_domain": 2_000_000,
    "proxy": {"n_params_target": 1_000_000, "tokens_per_proxy": 250_000_000, "block_size": 512},
    "sweep": {"n_mixtures": 50, "dirichlet_alpha": 1.0, "seed": 0},
    "target_weights": {"indic": 0.4, "code": 0.3, "maths": 0.2, "papers": 0.1, "web": 0.0}
}
with open(ROOT + "/src/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Scaffold ready at", ROOT)
print(json.dumps(config, indent=2))
import json
ROOT = "/content/regmix"
cfg = json.load(open(f"{ROOT}/src/config.json"))
cfg["hf_sources"]["papers"] = {
    "id": "neuralwork/arxiver", "config": None,
    "split": "train", "text_key": "markdown"
}
json.dump(cfg, open(f"{ROOT}/src/config.json", "w"), indent=2)
print("papers source ->", cfg["hf_sources"]["papers"])
import json
ROOT = "/content/regmix"
cfg = json.load(open(f"{ROOT}/src/config.json"))
cfg["hf_sources"]["code"] = {
    "id": "bigcode/the-stack-dedup", "config": "default",
    "split": "train", "text_key": "content", "data_dir": "data/python"
}
json.dump(cfg, open(f"{ROOT}/src/config.json", "w"), indent=2)
print("code source ->", cfg["hf_sources"]["code"])

Scaffold ready at /content/regmix
{
  "domains": [
    "web",
    "code",
    "maths",
    "indic",
    "papers"
  ],
  "hf_sources": {
    "web": {
      "id": "OptimalScale/ClimbMix",
      "config": null,
      "split": "train",
      "text_key": "text"
    },
    "code": {
      "id": "bigcode/the-stack-dedup",
      "config": "data/python",
      "split": "train",
      "text_key": "content"
    },
    "maths": {
      "id": "open-web-math/open-web-math",
      "config": null,
      "split": "train",
      "text_key": "text"
    },
    "papers": {
      "id": "togethercomputer/RedPajama-Data-1T",
      "config": "arxiv",
      "split": "train",
      "text_key": "text"
    }
  },
  "local_sources": {
    "indic": "/content/regmix/data_raw/indic"
  },
  "tokenizer": "bigscience/bloom",
  "tokens_per_domain": 60000000,
  "val_tokens_per_domain": 2000000,
  "proxy": {
    "n_params_target": 1000000,
    "tokens_per_proxy": 250000000,
    "block_size": 512
  },
  "sweep": {
    "n_mix

In [3]:
%%writefile /content/regmix/src/prepare_data.py
import os, json, sys, glob, numpy as np
from itertools import islice
from datasets import load_dataset
from transformers import AutoTokenizer
from tqdm import tqdm

ROOT = "/content/regmix"
cfg  = json.load(open(f"{ROOT}/src/config.json"))
BIN  = f"{ROOT}/data_bin"; os.makedirs(BIN, exist_ok=True)

tok = AutoTokenizer.from_pretrained(cfg["tokenizer"])
DTYPE = np.uint16 if tok.vocab_size < 65536 else np.uint32
print(f"tokenizer={cfg['tokenizer']} vocab={tok.vocab_size} dtype={DTYPE.__name__}")

def write_bin(path, token_iter, max_tokens):
    """Stream tokens into a growable memmap; stop at max_tokens."""
    buf = np.memmap(path, dtype=DTYPE, mode="w+", shape=(max_tokens,))
    n = 0
    pbar = tqdm(total=max_tokens, unit="tok", unit_scale=True, desc=os.path.basename(path))
    for ids in token_iter:
        if n >= max_tokens: break
        take = min(len(ids), max_tokens - n)
        buf[n:n+take] = ids[:take]; n += take; pbar.update(take)
    pbar.close(); buf.flush()
    # trim file to actual length
    actual = np.memmap(path, dtype=DTYPE, mode="r", shape=(n,))
    np.array(actual).tofile(path)
    print(f"  wrote {n:,} tokens -> {path}")
    return n

def hf_token_stream(src):
    ds = load_dataset(src["id"], src["config"], split=src["split"],
                      streaming=True, trust_remote_code=True)
    for ex in ds:
        txt = ex.get(src["text_key"])
        if txt:
            yield tok.encode(txt) + [tok.eos_token_id or 0]

def local_token_stream(folder):
    files = sorted(glob.glob(f"{folder}/**/*.txt", recursive=True)) + \
            sorted(glob.glob(f"{folder}/**/*.jsonl", recursive=True))
    assert files, f"No .txt/.jsonl found in {folder} — upload your Indic data there first."
    for fp in files:
        if fp.endswith(".jsonl"):
            for line in open(fp, encoding="utf-8"):
                try: txt = json.loads(line).get("text","")
                except: txt = ""
                if txt: yield tok.encode(txt) + [tok.eos_token_id or 0]
        else:
            txt = open(fp, encoding="utf-8").read()
            for para in txt.split("\n\n"):
                if para.strip(): yield tok.encode(para) + [tok.eos_token_id or 0]

def prepare_domain(name):
    train_tok = cfg["tokens_per_domain"]
    val_tok   = cfg["val_tokens_per_domain"]
    total     = train_tok + val_tok
    if name in cfg["local_sources"]:
        stream = local_token_stream(cfg["local_sources"][name])
    else:
        stream = hf_token_stream(cfg["hf_sources"][name])
    # write combined then split
    tmp = f"{BIN}/{name}_all.bin"
    n = write_bin(tmp, stream, total)
    arr = np.fromfile(tmp, dtype=DTYPE)
    n_val = min(val_tok, n // 10)
    arr[n_val:].tofile(f"{BIN}/{name}_train.bin")
    arr[:n_val].tofile(f"{BIN}/{name}_val.bin")
    os.remove(tmp)
    return {"name": name, "train": int(n - n_val), "val": int(n_val)}

if __name__ == "__main__":
    which = sys.argv[1:] or cfg["domains"]
    manifest = {}
    for name in which:
        print(f"\n=== {name} ===")
        manifest[name] = prepare_domain(name)
    # merge into any existing manifest
    mpath = f"{BIN}/manifest.json"
    old = json.load(open(mpath)) if os.path.exists(mpath) else {}
    old.update(manifest); json.dump(old, open(mpath,"w"), indent=2)
    print("\nManifest:", json.dumps(old, indent=2))

Writing /content/regmix/src/prepare_data.py


In [7]:
import re
p = "/content/regmix/src/prepare_data.py"
s = open(p).read()
s = s.replace(
    'ds = load_dataset(src["id"], src["config"], split=src["split"],\n                      streaming=True, trust_remote_code=True)',
    'ds = load_dataset(src["id"], src["config"], split=src["split"], streaming=True)'
)
open(p,"w").write(s)
print("patched: removed trust_remote_code")

patched: removed trust_remote_code


In [10]:
# smoke-test the pipeline on the two smaller/faster domains first
!cd /content/regmix && python src/prepare_data.py papers

tokenizer=bigscience/bloom vocab=250680 dtype=uint32

=== papers ===
papers_all.bin:   0% 0.00/62.0M [00:00<?, ?tok/s]
README.md: 100% 2.14k/2.14k [00:00<00:00, 8.21MB/s]
papers_all.bin: 100% 62.0M/62.0M [02:52<00:00, 360ktok/s]
  wrote 62,000,000 tokens -> /content/regmix/data_bin/papers_all.bin

Manifest: {
  "papers": {
    "name": "papers",
    "train": 60000000,
    "val": 2000000
  }
}
Fatal Python error: PyGILState_Release: thread state 0x7f2d8c05c1e0 must be current when releasing
Python runtime state: finalizing (tstate=0x0000000000b8a5b0)

Thread 0x00007f2f29dac280 (most recent call first):
  <no Python frame>

Extension modules: numpy._core._multiarray_umath, numpy._core._multiarray_tests, numpy.linalg._umath_linalg, zstandard.backend_c, pyarrow.lib, numpy.random._common, numpy.random.bit_generator, numpy.random._bounded_integers, numpy.random._mt19937, numpy.random.mtrand, numpy.random._philox, numpy.random._pcg64, numpy.random._sfc64, numpy.random._generator, pandas._libs.

In [12]:
from huggingface_hub import login
login(token = 'BdDW')

In [15]:
!cd /content/regmix && python src/prepare_data.py code

tokenizer=bigscience/bloom vocab=250680 dtype=uint32

=== code ===
code_all.bin:   0% 0.00/62.0M [00:00<?, ?tok/s]
Resolving data files:   0% 0/5142 [00:00<?, ?it/s]
Resolving data files:   1% 28/5142 [00:00<01:28, 57.54it/s]
Resolving data files: 100% 5142/5142 [00:00<00:00, 7655.16it/s]
code_all.bin: 100% 62.0M/62.0M [03:22<00:00, 306ktok/s]
  wrote 62,000,000 tokens -> /content/regmix/data_bin/code_all.bin

Manifest: {
  "papers": {
    "name": "papers",
    "train": 60000000,
    "val": 2000000
  },
  "code": {
    "name": "code",
    "train": 60000000,
    "val": 2000000
  }
}


In [16]:
import json
ROOT = "/content/regmix"
cfg = json.load(open(f"{ROOT}/src/config.json"))

# indic now comes from Sangraha verified, 4 languages, balanced
cfg.pop("local_sources", None)          # no longer local
cfg["indic_sangraha"] = {
    "id": "ai4bharat/sangraha",
    "subset": "verified",
    "langs": ["hin", "tam", "ben", "tel"],   # <-- your 4 codes
    "text_key": "text",
    "tokens_per_lang": 15_000_000            # 4 x 15M = 60M balanced
}
json.dump(cfg, open(f"{ROOT}/src/config.json","w"), indent=2)
print("indic ->", cfg["indic_sangraha"])

indic -> {'id': 'ai4bharat/sangraha', 'subset': 'verified', 'langs': ['hin', 'tam', 'ben', 'tel'], 'text_key': 'text', 'tokens_per_lang': 15000000}


In [17]:
p = "/content/regmix/src/prepare_data.py"
s = open(p).read()

# 1) new balanced multi-language Sangraha stream
inject = '''
def indic_sangraha_stream():
    ic = cfg["indic_sangraha"]
    import itertools
    gens = []
    for lang in ic["langs"]:
        ds = load_dataset(ic["id"], data_dir=f'{ic["subset"]}/{lang}',
                          split="train", streaming=True)
        gens.append((lang, iter(ds), ic["tokens_per_lang"]))
    # round-robin across languages, each capped at tokens_per_lang
    counts = {l: 0 for l, _, _ in gens}
    active = list(gens)
    while active:
        nxt = []
        for lang, g, cap in active:
            if counts[lang] >= cap:
                continue
            try:
                ex = next(g)
            except StopIteration:
                continue
            txt = ex.get(ic["text_key"])
            if txt:
                ids = tok.encode(txt) + [tok.eos_token_id or 0]
                counts[lang] += len(ids)
                yield ids
            nxt.append((lang, g, cap))
        active = nxt
    print("  indic per-language tokens:", counts)
'''
s = s.replace("def prepare_domain(name):", inject + "\ndef prepare_domain(name):")

# 2) route 'indic' to the new stream
s = s.replace(
    '    if name in cfg["local_sources"]:\n'
    '        stream = local_token_stream(cfg["local_sources"][name])\n'
    '    else:\n'
    '        stream = hf_token_stream(cfg["hf_sources"][name])',
    '    if name == "indic":\n'
    '        stream = indic_sangraha_stream()\n'
    '    else:\n'
    '        stream = hf_token_stream(cfg["hf_sources"][name])'
)
open(p,"w").write(s)
print("patched: indic now streams balanced from Sangraha verified")

patched: indic now streams balanced from Sangraha verified


In [18]:
!cd /content/regmix && python src/prepare_data.py indic

tokenizer=bigscience/bloom vocab=250680 dtype=uint32

=== indic ===
indic_all.bin:   0% 0.00/62.0M [00:00<?, ?tok/s]
README.md: 100% 10.9k/10.9k [00:00<00:00, 14.1MB/s]

Resolving data files: 100% 100/100 [00:00<00:00, 30310.04it/s]

Resolving data files: 100% 53/53 [00:00<00:00, 264640.61it/s]

Resolving data files: 100% 77/77 [00:00<00:00, 265157.15it/s]

Resolving data files: 100% 41/41 [00:00<00:00, 259376.27it/s]
indic_all.bin:  97% 60.0M/62.0M [05:36<00:09, 219ktok/s]  indic per-language tokens: {'hin': 15000035, 'tam': 15000102, 'ben': 15000063, 'tel': 15000379}
indic_all.bin:  97% 60.0M/62.0M [05:37<00:11, 178ktok/s]
  wrote 60,000,579 tokens -> /content/regmix/data_bin/indic_all.bin

Manifest: {
  "papers": {
    "name": "papers",
    "train": 60000000,
    "val": 2000000
  },
  "code": {
    "name": "code",
    "train": 60000000,
    "val": 2000000
  },
  "indic": {
    "name": "indic",
    "train": 58000579,
    "val": 2000000
  }
}


In [23]:
import os, numpy as np, json
LOCAL = "/content/regmix/data_bin"
DTYPE = np.uint32  # BLOOM vocab > 65536

print(f"{'domain':8s} {'train tok':>14s} {'val tok':>12s}")
for name in ["web","code","maths","indic","papers"]:
    tr, va = f"{LOCAL}/{name}_train.bin", f"{LOCAL}/{name}_val.bin"
    if os.path.exists(tr) and os.path.exists(va):
        nt = np.fromfile(tr, dtype=DTYPE).size
        nv = np.fromfile(va, dtype=DTYPE).size
        print(f"{name:8s} {nt:14,} {nv:12,}")
    else:
        print(f"{name:8s} {'MISSING':>14s}")

domain        train tok      val tok
web          60,000,000    2,000,000
code         60,000,000    2,000,000
maths        60,000,000    2,000,000
indic        58,000,579    2,000,000
papers       60,000,000    2,000,000


In [20]:
import json
cfg = json.load(open("/content/regmix/src/config.json"))
print("web source ->", cfg["hf_sources"].get("web"))

web source -> {'id': 'OptimalScale/ClimbMix', 'config': None, 'split': 'train', 'text_key': 'text'}


In [21]:
!cd /content/regmix && python src/prepare_data.py web

tokenizer=bigscience/bloom vocab=250680 dtype=uint32

=== web ===
web_all.bin:   0% 0.00/62.0M [00:00<?, ?tok/s]
README.md: 100% 1.84k/1.84k [00:00<00:00, 5.05MB/s]

Resolving data files: 100% 100/100 [00:00<00:00, 307725.90it/s]
web_all.bin: 100% 62.0M/62.0M [03:46<00:00, 274ktok/s] 
  wrote 62,000,000 tokens -> /content/regmix/data_bin/web_all.bin

Manifest: {
  "papers": {
    "name": "papers",
    "train": 60000000,
    "val": 2000000
  },
  "code": {
    "name": "code",
    "train": 60000000,
    "val": 2000000
  },
  "indic": {
    "name": "indic",
    "train": 58000579,
    "val": 2000000
  },
  "web": {
    "name": "web",
    "train": 60000000,
    "val": 2000000
  }
}


In [22]:
%%writefile /content/regmix/src/model.py
import os, json, math, numpy as np, torch, torch.nn as nn
from torch.nn import functional as F

ROOT = "/content/regmix"
cfg  = json.load(open(f"{ROOT}/src/config.json"))
BIN  = os.environ.get("REGMIX_BIN", f"{ROOT}/data_bin")
DOMAINS = cfg["domains"]
DTYPE = np.uint32
DEVICE = "cuda"

# ---------- model (tiny GPT, weight-tied) ----------
class Block(nn.Module):
    def __init__(s, d, h):
        super().__init__()
        s.ln1, s.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.attn = nn.MultiheadAttention(d, h, batch_first=True)
        s.mlp = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(s, x, mask):
        a,_ = s.attn(s.ln1(x), s.ln1(x), s.ln1(x), attn_mask=mask, need_weights=False)
        x = x + a
        return x + s.mlp(s.ln2(x))

class TinyGPT(nn.Module):
    def __init__(s, vocab, d=128, h=4, L=4, block=512):
        super().__init__()
        s.block = block
        s.tok = nn.Embedding(vocab, d)
        s.pos = nn.Embedding(block, d)
        s.blocks = nn.ModuleList([Block(d, h) for _ in range(L)])
        s.lnf = nn.LayerNorm(d)
        s.head = nn.Linear(d, vocab, bias=False)
        s.head.weight = s.tok.weight            # weight tying
        s.register_buffer("mask", torch.triu(torch.ones(block, block)*float("-inf"), 1))
    def forward(s, idx, targets=None):
        T = idx.size(1)
        x = s.tok(idx) + s.pos(torch.arange(T, device=idx.device))
        for b in s.blocks: x = b(x, s.mask[:T,:T])
        logits = s.head(s.lnf(x))
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

def count_non_embed(m):
    emb = m.tok.weight.numel() + m.pos.weight.numel()
    return sum(p.numel() for p in m.parameters()) - emb

# ---------- memmap loaders ----------
_mm = {}
def _get(name, split):
    k=(name,split)
    if k not in _mm:
        _mm[k]=np.memmap(f"{BIN}/{name}_{split}.bin", dtype=DTYPE, mode="r")
    return _mm[k]

def get_batch(mixture, split, bs, block):
    """Draw `bs` sequences; each sequence's domain is sampled from `mixture`."""
    counts = np.random.multinomial(bs, mixture)
    xs, ys = [], []
    for name, c in zip(DOMAINS, counts):
        if c==0: continue
        arr=_get(name, split); n=len(arr)
        starts=np.random.randint(0, n-block-1, size=c)
        for st in starts:
            chunk=arr[st:st+block+1].astype(np.int64)
            xs.append(chunk[:-1]); ys.append(chunk[1:])
    X=torch.from_numpy(np.stack(xs)).to(DEVICE)
    Y=torch.from_numpy(np.stack(ys)).to(DEVICE)
    return X, Y

# ---------- train one proxy from scratch ----------
def train_one_proxy(mixture, tokens=None, bs=64, lr=3e-3, log_every=200):
    torch.manual_seed(0); np.random.seed(0)
    p=cfg["proxy"]; block=p["block_size"]
    tokens = tokens or p["tokens_per_proxy"]
    vocab = int(max(_get(d,"train").max() for d in DOMAINS))+1
    model=TinyGPT(vocab, block=block).to(DEVICE)
    if os.environ.get("PRINT_PARAMS"):
        print(f"non-embed params: {count_non_embed(model):,} | total: {sum(x.numel() for x in model.parameters()):,}")
    opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    steps=tokens // (bs*block)
    model.train()
    for step in range(steps):
        X,Y=get_batch(mixture,"train",bs,block)
        _,loss=model(X,Y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        if step % log_every==0:
            print(f"  step {step}/{steps} loss {loss.item():.3f}", flush=True)
    # per-domain val loss (pure domain mixtures)
    model.eval(); vloss={}
    with torch.no_grad():
        for i,name in enumerate(DOMAINS):
            onehot=np.zeros(len(DOMAINS)); onehot[i]=1.0
            ls=[]
            for _ in range(20):
                X,Y=get_batch(onehot,"val",bs,block)
                _,l=model(X,Y); ls.append(l.item())
            vloss[name]=float(np.mean(ls))
    return vloss

if __name__=="__main__":
    os.environ["PRINT_PARAMS"]="1"
    print("Domains:", DOMAINS)
    vl=train_one_proxy([0.2]*5, tokens=2_000_000)   # tiny smoke test
    print("val losses:", json.dumps(vl, indent=2))

Writing /content/regmix/src/model.py


In [25]:
p = "/content/regmix/src/model.py"
s = open(p).read()

# 1) train loop: micro-batch + grad-accum + bf16 autocast
s = s.replace(
'''def train_one_proxy(mixture, tokens=None, bs=64, lr=3e-3, log_every=200):
    torch.manual_seed(0); np.random.seed(0)
    p=cfg["proxy"]; block=p["block_size"]
    tokens = tokens or p["tokens_per_proxy"]
    vocab = int(max(_get(d,"train").max() for d in DOMAINS))+1
    model=TinyGPT(vocab, block=block).to(DEVICE)
    if os.environ.get("PRINT_PARAMS"):
        print(f"non-embed params: {count_non_embed(model):,} | total: {sum(x.numel() for x in model.parameters()):,}")
    opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    steps=tokens // (bs*block)
    model.train()
    for step in range(steps):
        X,Y=get_batch(mixture,"train",bs,block)
        _,loss=model(X,Y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        if step % log_every==0:
            print(f"  step {step}/{steps} loss {loss.item():.3f}", flush=True)''',
'''def train_one_proxy(mixture, tokens=None, micro_bs=16, accum=4, lr=3e-3, log_every=200):
    torch.manual_seed(0); np.random.seed(0)
    p=cfg["proxy"]; block=p["block_size"]
    tokens = tokens or p["tokens_per_proxy"]
    vocab = int(max(_get(d,"train").max() for d in DOMAINS))+1
    model=TinyGPT(vocab, block=block).to(DEVICE)
    if os.environ.get("PRINT_PARAMS"):
        print(f"non-embed params: {count_non_embed(model):,} | total: {sum(x.numel() for x in model.parameters()):,}")
    opt=torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95), weight_decay=0.1)
    eff_bs = micro_bs*accum
    steps = tokens // (eff_bs*block)
    model.train()
    for step in range(steps):
        opt.zero_grad(set_to_none=True)
        for _ in range(accum):
            X,Y=get_batch(mixture,"train",micro_bs,block)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                _,loss=model(X,Y)
            (loss/accum).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        if step % log_every==0:
            print(f"  step {step}/{steps} loss {loss.item():.3f}", flush=True)'''
)

# 2) val loop: bf16 autocast + smaller batch, and free cache between proxies
s = s.replace(
'''    model.eval(); vloss={}
    with torch.no_grad():
        for i,name in enumerate(DOMAINS):
            onehot=np.zeros(len(DOMAINS)); onehot[i]=1.0
            ls=[]
            for _ in range(20):
                X,Y=get_batch(onehot,"val",bs,block)
                _,l=model(X,Y); ls.append(l.item())
            vloss[name]=float(np.mean(ls))
    return vloss''',
'''    model.eval(); vloss={}
    with torch.no_grad():
        for i,name in enumerate(DOMAINS):
            onehot=np.zeros(len(DOMAINS)); onehot[i]=1.0
            ls=[]
            for _ in range(20):
                X,Y=get_batch(onehot,"val",16,block)
                with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                    _,l=model(X,Y)
                ls.append(l.item())
            vloss[name]=float(np.mean(ls))
    del model, opt
    torch.cuda.empty_cache()
    return vloss'''
)
open(p,"w").write(s)
print("patched: bf16 autocast + micro-batch/grad-accum + cache cleanup")

patched: bf16 autocast + micro-batch/grad-accum + cache cleanup


In [27]:
p = "/content/regmix/src/model.py"
s = open(p).read()
s = s.replace(
    '        s.register_buffer("mask", torch.triu(torch.ones(block, block)*float("-inf"), 1))',
    '''        s.register_buffer("mask", torch.triu(torch.ones(block, block)*float("-inf"), 1))
        s.apply(s._init)
    def _init(s, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)'''
)
# keep cross-entropy in fp32 for numerical stability on the wide vocab
s = s.replace(
    '        logits = s.head(s.lnf(x))\n        if targets is None: return logits, None\n        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))',
    '        logits = s.head(s.lnf(x))\n        if targets is None: return logits, None\n        loss = F.cross_entropy(logits.view(-1, logits.size(-1)).float(), targets.view(-1))'
)
open(p,"w").write(s)
print("patched: proper init + fp32 cross-entropy")

patched: proper init + fp32 cross-entropy


In [26]:
!cd /content/regmix && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/model.py

Domains: ['web', 'code', 'maths', 'indic', 'papers']
non-embed params: 793,344 | total: 32,945,920
  step 0/61 loss 85.041
val losses: {
  "web": 10.777905702590942,
  "code": 13.981032752990723,
  "maths": 11.777730751037598,
  "indic": 15.622356700897218,
  "papers": 10.576459693908692
}


In [ ]:
import os, json, sys, importlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
sys.path.insert(0, "/content/regmix/src")
import model as M; importlib.reload(M)

# 40-step run, log EVERY step to see the descent
vl = M.train_one_proxy([0.2]*5, tokens=1_400_000, micro_bs=16, accum=4, log_every=1)
print(json.dumps(vl, indent=2))

In [29]:
%%writefile /content/regmix/src/sweep.py
import os, json, sys, time, numpy as np
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0, "/content/regmix/src")
import model as M

ROOT="/content/regmix"; cfg=json.load(open(f"{ROOT}/src/config.json"))
DOMAINS=cfg["domains"]; RUNS=f"{ROOT}/runs"; os.makedirs(RUNS,exist_ok=True)
LOG=f"{RUNS}/sweep_log.jsonl"
TW=cfg["target_weights"]

def target_metric(vloss):
    return sum(TW[d]*vloss[d] for d in DOMAINS)

def sample_mixtures(n, alpha, seed):
    rng=np.random.default_rng(seed)
    mix=rng.dirichlet([alpha]*len(DOMAINS), size=n)
    # add a few near-extreme mixtures so the regressor sees the edges
    for i in range(len(DOMAINS)):
        e=np.full(len(DOMAINS),0.02); e[i]=1-0.02*(len(DOMAINS)-1); mix=np.vstack([mix,e])
    return mix

def done_ids():
    if not os.path.exists(LOG): return set()
    return {json.loads(l)["id"] for l in open(LOG)}

def run_sweep():
    s=cfg["sweep"]
    mix=sample_mixtures(s["n_mixtures"], s["dirichlet_alpha"], s["seed"])
    done=done_ids()
    print(f"{len(mix)} mixtures total, {len(done)} already done")
    for i,m in enumerate(mix):
        if i in done:
            continue
        t0=time.time()
        vloss=M.train_one_proxy(m.tolist(), log_every=999999)  # quiet per-proxy
        rec={"id":i,"mixture":m.tolist(),"vloss":vloss,
             "target":target_metric(vloss),"sec":round(time.time()-t0,1)}
        with open(LOG,"a") as f: f.write(json.dumps(rec)+"\n")   # atomic append
        print(f"[{i+1}/{len(mix)}] target={rec['target']:.3f} ({rec['sec']}s)", flush=True)
    print("sweep complete.")

def fit_and_search():
    import lightgbm as lgb
    from scipy.stats import spearmanr
    recs=[json.loads(l) for l in open(LOG)]
    X=np.array([r["mixture"] for r in recs]); y=np.array([r["target"] for r in recs])
    # holdout to sanity-check the predictor
    n=len(X); idx=np.random.default_rng(0).permutation(n)
    cut=int(n*0.8); tr,te=idx[:cut],idx[cut:]
    dtrain=lgb.Dataset(X[tr],y[tr])
    params=dict(objective="regression",num_leaves=15,min_data_in_leaf=5,
                learning_rate=0.05,lambda_l1=0.1,lambda_l2=0.1,verbose=-1)
    gbm=lgb.train(params,dtrain,num_boost_round=500)
    pred=gbm.predict(X[te])
    rho=spearmanr(pred,y[te]).correlation
    print(f"held-out rank corr (Spearman): {rho:.3f}  (want > 0.7)")
    # refit on all data, search 100k candidates
    gbm=lgb.train(params,lgb.Dataset(X,y),num_boost_round=500)
    cand=np.random.default_rng(1).dirichlet([1.0]*len(DOMAINS),size=100_000)
    p=gbm.predict(cand)
    best=cand[p.argmin()]
    print("\nOPTIMAL MIXTURE (minimizes weighted target):")
    for d,w in zip(DOMAINS,best): print(f"  {d:8s} {w:.3f}")
    json.dump({"domains":DOMAINS,"optimal":best.tolist()},
              open(f"{RUNS}/optimal_mixture.json","w"),indent=2)
    return best

if __name__=="__main__":
    if sys.argv[1:]==["search"]:
        fit_and_search()
    else:
        run_sweep()
        fit_and_search()

Writing /content/regmix/src/sweep.py


In [33]:
p = "/content/regmix/src/model.py"
s = open(p).read()

# Replace nn.MultiheadAttention block with a fused-SDPA causal attention
s = s.replace(
'''class Block(nn.Module):
    def __init__(s, d, h):
        super().__init__()
        s.ln1, s.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.attn = nn.MultiheadAttention(d, h, batch_first=True)
        s.mlp = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(s, x, mask):
        a,_ = s.attn(s.ln1(x), s.ln1(x), s.ln1(x), attn_mask=mask, need_weights=False)
        x = x + a
        return x + s.mlp(s.ln2(x))''',
'''class Block(nn.Module):
    def __init__(s, d, h):
        super().__init__()
        s.h = h; s.d = d
        s.ln1, s.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.qkv = nn.Linear(d, 3*d)
        s.proj = nn.Linear(d, d)
        s.mlp = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(s, x, mask=None):
        B,T,D = x.shape
        q,k,v = s.qkv(s.ln1(x)).split(D, dim=2)
        q=q.view(B,T,s.h,D//s.h).transpose(1,2)
        k=k.view(B,T,s.h,D//s.h).transpose(1,2)
        v=v.view(B,T,s.h,D//s.h).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True)  # fused Flash kernel
        a=a.transpose(1,2).contiguous().view(B,T,D)
        x = x + s.proj(a)
        return x + s.mlp(s.ln2(x))'''
)

# mask buffer no longer needed; forward passes no mask
s = s.replace('        for b in s.blocks: x = b(x, s.mask[:T,:T])',
              '        for b in s.blocks: x = b(x)')
s = s.replace('        s.register_buffer("mask", torch.triu(torch.ones(block, block)*float("-inf"), 1))\n', '')

# bigger micro-batch now that SDPA frees memory: 48 micro x 2 accum = 96 eff
s = s.replace('def train_one_proxy(mixture, tokens=None, micro_bs=16, accum=4, lr=3e-3, log_every=200):',
              'def train_one_proxy(mixture, tokens=None, micro_bs=48, accum=2, lr=3e-3, log_every=200):')
open(p,"w").write(s)
print("patched: fused SDPA attention + micro_bs 48")

patched: fused SDPA attention + micro_bs 48


In [4]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

free: 4.2 GB


In [2]:
import os, numpy as np
LOCAL="/content/regmix/data_bin"; DTYPE=np.uint32
for n in ["web","code","maths","indic","papers"]:
    tr=f"{LOCAL}/{n}_train.bin"
    print(f"{n:8s}", f"{np.fromfile(tr,dtype=DTYPE).size:,}" if os.path.exists(tr) else "MISSING")

web      60,000,000
code     60,000,000
maths    60,000,000
indic    58,000,579
papers   60,000,000


In [5]:
s = open("/content/regmix/src/model.py").read()
print("micro_bs=48 present:", "micro_bs=48" in s)
print("micro_bs=32 present:", "micro_bs=32" in s)
print("chunked CE present: ", "chunk = 4096" in s)
print("SDPA present:        ", "scaled_dot_product_attention" in s)

micro_bs=48 present: True
micro_bs=32 present: False
chunked CE present:  False
SDPA present:         True


In [6]:
p = "/content/regmix/src/model.py"
s = open(p).read()

if "chunk = 4096" not in s:
    s = s.replace(
'''        logits = s.head(s.lnf(x))
        if targets is None: return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)).float(), targets.view(-1))
        return logits, loss''',
'''        h = s.lnf(x)
        if targets is None:
            return s.head(h), None
        h = h.view(-1, h.size(-1)); t = targets.view(-1)
        chunk = 4096; total = 0.0; n = h.size(0)
        for i in range(0, n, chunk):
            lg = s.head(h[i:i+chunk]).float()
            total = total + F.cross_entropy(lg, t[i:i+chunk], reduction="sum")
        return None, total / n''')

import re
s = re.sub(r'def train_one_proxy\(mixture, tokens=None, micro_bs=\d+, accum=\d+,',
           'def train_one_proxy(mixture, tokens=None, micro_bs=32, accum=3,', s)
open(p,"w").write(s)

s2 = open(p).read()
print("micro_bs=32:", "micro_bs=32" in s2, "| chunked CE:", "chunk = 4096" in s2, "| SDPA:", "scaled_dot_product_attention" in s2)

micro_bs=32: True | chunked CE: True | SDPA: True


In [1]:
s = open("/content/regmix/src/model.py").read()
i = s.find("def forward(s, idx")
print(s[i:i+700])

def forward(s, idx, targets=None):
        T = idx.size(1)
        x = s.tok(idx) + s.pos(torch.arange(T, device=idx.device))
        for b in s.blocks: x = b(x)
        h = s.lnf(x)
        if targets is None:
            return s.head(h), None
        h = h.view(-1, h.size(-1)); t = targets.view(-1)
        chunk = 4096; total = 0.0; n = h.size(0)
        for i in range(0, n, chunk):
            lg = s.head(h[i:i+chunk]).float()
            total = total + F.cross_entropy(lg, t[i:i+chunk], reduction="sum")
        return None, total / n

def count_non_embed(m):
    emb = m.tok.weight.numel() + m.pos.weight.numel()
    return sum(p.numel() for p in m.parameters()) - emb

# ---------- memmap


In [3]:
!nvidia-smi

Fri Jul 31 00:35:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             52W /  400W |   39439MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
import os, signal
# kill the zombie holding 35 GB (PID from the error / nvidia-smi)
try:
    os.kill(21220, signal.SIGKILL)
    print("killed 21220")
except Exception as e:
    print("couldn't kill:", e)

killed 21220


In [5]:
# 1. verify the file on disk is the patched version
s = open("/content/regmix/src/model.py").read()
print("micro_bs=32:", "micro_bs=32" in s, "| chunked CE:", "chunk = 4096" in s, "| SDPA:", "scaled_dot_product_attention" in s)
# must be all True

micro_bs=32: True | chunked CE: True | SDPA: True


In [6]:
# 2. confirm GPU is actually empty in the fresh session
import torch
print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")   # want ~42

free: 39.3 GB


In [18]:
import json
c = json.load(open("/content/regmix/src/config.json"))
c["proxy"]["tokens_per_proxy"] = 50_000_000
c["sweep"]["n_mixtures"] = 32
json.dump(c, open("/content/regmix/src/config.json","w"), indent=2)
print("cut to:", c["proxy"]["tokens_per_proxy"], "tokens,", c["sweep"]["n_mixtures"], "mixtures")

cut to: 50000000 tokens, 32 mixtures


In [19]:
# bump batch: micro_bs 32 -> 64, accum 3 -> 2  (eff batch 128, fewer steps, chunked CE keeps memory safe)
p="/content/regmix/src/model.py"; s=open(p).read()
import re
s=re.sub(r'def train_one_proxy\(mixture, tokens=None, micro_bs=\d+, accum=\d+,',
         'def train_one_proxy(mixture, tokens=None, micro_bs=64, accum=2,', s)
open(p,"w").write(s)
print("micro_bs=64, accum=2:", "micro_bs=64" in open(p).read())

micro_bs=64, accum=2: True


In [20]:
import subprocess, os
out = subprocess.check_output(["nvidia-smi","--query-compute-apps=pid","--format=csv,noheader"]).decode()
pids = [int(p) for p in out.split() if p.strip().isdigit()]
me = os.getpid()
for pid in pids:
    if pid != me:
        try: os.kill(pid, 9); print("killed", pid)
        except Exception as e: print("skip", pid, e)
import torch, gc; gc.collect(); torch.cuda.empty_cache()
print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

free: 41.4 GB


In [21]:
p="/content/regmix/src/model.py"; s=open(p).read()
import re
s=re.sub(r'def train_one_proxy\(mixture, tokens=None, micro_bs=\d+, accum=\d+,',
         'def train_one_proxy(mixture, tokens=None, micro_bs=48, accum=2,', s)
open(p,"w").write(s)
print("micro_bs=48:", "micro_bs=48" in open(p).read())

micro_bs=48: True


In [22]:
import os, sys, time, gc, torch, importlib
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
gc.collect(); torch.cuda.empty_cache()
sys.path.insert(0,"/content/regmix/src")
import model as M; importlib.reload(M)
t0=time.time()
vl=M.train_one_proxy([0.2]*5, log_every=500)
print(f"one proxy: {(time.time()-t0):.0f}s", vl)

  step 0/1017 loss 12.464
  step 500/1017 loss 5.993
  step 1000/1017 loss 5.483
one proxy: 560s {'web': 5.573098158836364, 'code': 4.879901242256165, 'maths': 6.085808324813843, 'indic': 6.550990319252014, 'papers': 4.670271682739258}


In [23]:
!zip -r folder_name.zip regmix/

  adding: regmix/ (stored 0%)
  adding: regmix/data_raw/ (stored 0%)
  adding: regmix/data_raw/indic/ (stored 0%)
  adding: regmix/data_bin/ (stored 0%)
  adding: regmix/data_bin/maths_train.bin (deflated 64%)
  adding: regmix/data_bin/manifest.json (deflated 69%)
  adding: regmix/data_bin/indic_train.bin (deflated 51%)
  adding: regmix/data_bin/indic_val.bin (deflated 50%)
  adding: regmix/data_bin/papers_val.bin (deflated 72%)
  adding: regmix/data_bin/maths_val.bin (deflated 64%)
  adding: regmix/data_bin/web_train.bin (deflated 61%)
  adding: regmix/data_bin/code_train.bin (deflated 78%)
  adding: regmix/data_bin/code_val.bin (deflated 79%)
  adding: regmix/data_bin/web_val.bin (deflated 59%)
  adding: regmix/data_bin/papers_train.bin (deflated 72%)
  adding: regmix/runs/ (stored 0%)
  adding: regmix/src/ (stored 0%)
  adding: regmix/src/config.json (deflated 63%)
  adding: regmix/src/prepare_data.py (deflated 59%)
  adding: regmix/src/sweep.py (deflated 51%)
  adding: regmix/src/_

In [24]:
from google.colab import files
files.download('folder_name.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
!cd /content/regmix && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/sweep.py

37 mixtures total, 0 already done
  step 0/1017 loss 12.467
[1/37] target=7.540 (560.7s)
  step 0/1017 loss 12.461
[2/37] target=5.934 (559.1s)
  step 0/1017 loss 12.461
[3/37] target=8.064 (559.1s)
  step 0/1017 loss 12.467
[4/37] target=6.010 (559.1s)
  step 0/1017 loss 12.461
[5/37] target=6.103 (559.1s)
  step 0/1017 loss 12.462
[6/37] target=5.813 (559.2s)
  step 0/1017 loss 12.463
[7/37] target=5.664 (559.1s)
  step 0/1017 loss 12.467
[8/37] target=5.711 (558.9s)
  step 0/1017 loss 12.462
[9/37] target=5.850 (559.1s)
  step 0/1017 loss 12.461
[10/37] target=6.208 (559.1s)
  step 0/1017 loss 12.462
[11/37] target=6.595 (559.1s)
  step 0/1017 loss 12.462
[12/37] target=5.865 (559.1s)
  step 0/1017 loss 12.466
[13/37] target=5.589 (559.2s)
  step 0/1017 loss 12.463
[14/37] target=5.838 (559.1s)
  step 0/1017 loss 12.467
[15/37] target=5.687 (558.9s)
  step 0/1017 loss 12.463
[16/37] target=7.073 (559.1s)
  step 0/1017 loss 12.461
[17/37] target=5.865 (559.1s)
  step 0/1017 loss 12.4

In [36]:
import json, numpy as np, lightgbm as lgb
recs=[json.loads(l) for l in open("/content/regmix/runs/sweep_log.jsonl")]
DOMAINS=["web","code","maths","indic","papers"]
X=np.array([r["mixture"] for r in recs])
VL={d:np.array([r["vloss"][d] for r in recs]) for d in DOMAINS}

center={"web":0.60,"code":0.05,"maths":0.10,"papers":0.10,"indic":0.15}
rng=np.random.default_rng(0)
params=dict(objective="regression",num_leaves=15,min_data_in_leaf=5,
            learning_rate=0.05,lambda_l1=0.1,lambda_l2=0.1,verbose=-1)
cand=rng.dirichlet([1.0]*5,size=100_000)

opt=[]
for _ in range(200):
    # jitter each target weight +/-30% (relative), renormalize to sum 1
    tw={d: max(0.0, center[d]*(1+rng.uniform(-0.3,0.3))) for d in DOMAINS}
    s=sum(tw.values()); tw={d:v/s for d,v in tw.items()}
    y=sum(tw[d]*VL[d] for d in DOMAINS)
    gbm=lgb.train(params, lgb.Dataset(X,y), num_boost_round=300)
    opt.append(cand[gbm.predict(cand).argmin()])
opt=np.array(opt)

print("Optimal mixture under jittered targets (200 draws):")
print(f"{'domain':8s} {'mean':>7s} {'std':>7s} {'p10':>7s} {'p90':>7s}")
for i,d in enumerate(DOMAINS):
    c=opt[:,i]
    print(f"{d:8s} {c.mean():7.3f} {c.std():7.3f} {np.percentile(c,10):7.3f} {np.percentile(c,90):7.3f}")

# also:

[backup] 53 proxies saved to Drive
Optimal mixture under jittered targets (200 draws):
domain      mean     std     p10     p90
web        0.442   0.128   0.280   0.562
code       0.041   0.028   0.022   0.063
maths      0.232   0.187   0.028   0.431
indic      0.166   0.027   0.137   0.203
papers     0.119   0.079   0.042   0.216


In [28]:
# mount Drive if not already
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/regmix_data', exist_ok=True)
print("Drive ready")

Mounted at /content/drive
Drive ready


In [29]:
import threading, shutil, time, os
SRC="/content/regmix/runs/sweep_log.jsonl"
DST="/content/drive/MyDrive/regmix_data/sweep_log.jsonl"

def backup_loop():
    while True:
        try:
            if os.path.exists(SRC):
                shutil.copy(SRC, DST)
                n=sum(1 for _ in open(SRC))
                print(f"[backup] {n} proxies saved to Drive", flush=True)
        except Exception as e:
            print("[backup] err:", e, flush=True)
        time.sleep(120)

threading.Thread(target=backup_loop, daemon=True).start()
print("auto-backup started — copies log to Drive every 2 min")

auto-backup started — copies log to Drive every 2 min


In [30]:
%%writefile /content/regmix/src/refine.py
import os, json, sys, time, numpy as np
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"/content/regmix/src")
import model as M
ROOT="/content/regmix"; cfg=json.load(open(f"{ROOT}/src/config.json"))
DOMAINS=cfg["domains"]; LOG=f"{ROOT}/runs/sweep_log.jsonl"; TW=cfg["target_weights"]

# center on your recipe (order = DOMAINS = web,code,maths,indic,papers)
CENTER = {"web":0.60,"code":0.10,"maths":0.10,"indic":0.15,"papers":0.05}
center = np.array([CENTER[d] for d in DOMAINS])
N_NEW  = 16
CONC   = 200.0   # high concentration = tight cluster around center

def target_metric(vl): return sum(TW[d]*vl[d] for d in DOMAINS)
def done_ids():
    if not os.path.exists(LOG): return set()
    return {json.loads(l)["id"] for l in open(LOG)}

# Dirichlet centered on `center`: alpha = center * CONC
rng=np.random.default_rng(100)                       # new seed, distinct from sweep
new_mix = rng.dirichlet(center*CONC, size=N_NEW)

done = done_ids()
start_id = 1000                                      # refine ids start at 1000 to avoid clashing with sweep ids
p

Writing /content/regmix/src/refine.py


In [31]:
import numpy as np
center=np.array([0.60,0.10,0.10,0.15,0.05])
rng=np.random.default_rng(100)
s=rng.dirichlet(center*200.0, size=16)
print("mean:", np.round(s.mean(0),3))   # should be ~[0.60,0.10,0.10,0.15,0.05]
print("std: ", np.round(s.std(0),3))    # should be small, ~0.02-0.04
print("min: ", np.round(s.min(0),3), "max:", np.round(s.max(0),3))

mean: [0.595 0.098 0.099 0.15  0.057]
std:  [0.042 0.021 0.02  0.023 0.017]
min:  [0.531 0.066 0.066 0.117 0.032] max: [0.671 0.151 0.132 0.195 0.099]


In [33]:
%%writefile /content/regmix/src/refine.py
import os, json, sys, time, numpy as np
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"/content/regmix/src")
import model as M
ROOT="/content/regmix"; cfg=json.load(open(f"{ROOT}/src/config.json"))
DOMAINS=cfg["domains"]; LOG=f"{ROOT}/runs/sweep_log.jsonl"; TW=cfg["target_weights"]

CENTER = {"web":0.60,"code":0.10,"maths":0.10,"indic":0.15,"papers":0.05}
center = np.array([CENTER[d] for d in DOMAINS])
N_NEW  = 16
CONC   = 200.0

def target_metric(vl): return sum(TW[d]*vl[d] for d in DOMAINS)
def done_ids():
    if not os.path.exists(LOG): return set()
    return {json.loads(l)["id"] for l in open(LOG)}

rng=np.random.default_rng(100)
new_mix = rng.dirichlet(center*CONC, size=N_NEW)
done = done_ids()
start_id = 1000
print(f"{N_NEW} refinement proxies around {CENTER}, conc={CONC}")
for j in range(N_NEW):
    jid = start_id + j
    if jid in done:
        print(f"  skip {jid} (done)"); continue
    m = new_mix[j]
    t0=time.time()
    vl = M.train_one_proxy(m.tolist(), log_every=10**9)
    rec={"id":jid,"mixture":m.tolist(),"vloss":vl,
         "target":target_metric(vl),"sec":round(time.time()-t0,1),"phase":"refine"}
    with open(LOG,"a") as f: f.write(json.dumps(rec)+"\n")
    print(f"[refine {j+1}/{N_NEW}] id={jid} target={rec['target']:.4f} "
          f"mix={[round(x,3) for x in m.tolist()]} ({rec['sec']}s)", flush=True)
print("refinement complete.")

Overwriting /content/regmix/src/refine.py


In [34]:
!wc -l /content/regmix/runs/sweep_log.jsonl   # still 37?
!cd /content/regmix && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/refine.py

37 /content/regmix/runs/sweep_log.jsonl
16 refinement proxies around {'web': 0.6, 'code': 0.1, 'maths': 0.1, 'indic': 0.15, 'papers': 0.05}, conc=200.0
  step 0/1017 loss 12.462
[backup] 37 proxies saved to Drive
[backup] 37 proxies saved to Drive
[backup] 37 proxies saved to Drive
[backup] 37 proxies saved to Drive
[backup] 37 proxies saved to Drive
[refine 1/16] id=1000 target=5.9383 mix=[0.559, 0.121, 0.082, 0.175, 0.063] (561.0s)
  step 0/1017 loss 12.459
[backup] 38 proxies saved to Drive
[backup] 38 proxies saved to Drive
[backup] 38 proxies saved to Drive
[backup] 38 proxies saved to Drive
[refine 2/16] id=1001 target=6.0900 mix=[0.671, 0.091, 0.066, 0.12, 0.052] (559.2s)
  step 0/1017 loss 12.463
[backup] 39 proxies saved to Drive
[backup] 39 proxies saved to Drive
[backup] 39 proxies saved to Drive
[backup] 39 proxies saved to Drive
[backup] 39 proxies saved to Drive
[refine 3/16] id=1002 target=5.9755 mix=[0.531, 0.091, 0.132, 0.195, 0.051] (559.3s)
  step 0/1017 loss 12.460


In [ ]:
!cp /content/regmix/runs/sweep_log.jsonl /content/drive/MyDrive/regmix_data/ 2>/dev/null && echo "backed up $(wc -l < /content/regmix/runs/sweep_log.jsonl) proxies" || echo "mount Drive first"

In [42]:
%%writefile /content/regmix/src/sweep_more.py
import os, json, sys, time, numpy as np
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
sys.path.insert(0,"/content/regmix/src")
import model as M
ROOT="/content/regmix"; cfg=json.load(open(f"{ROOT}/src/config.json"))
DOMAINS=cfg["domains"]; LOG=f"{ROOT}/runs/sweep_log.jsonl"; TW=cfg["target_weights"]

N_NEW   = 20
ALPHA   = 1.0        # broad Dirichlet, same as original sweep
START_ID = 2000      # distinct from sweep (0-36) and refine (1000-1015)

def target_metric(vl): return sum(TW[d]*vl[d] for d in DOMAINS)
def done_ids():
    if not os.path.exists(LOG): return set()
    return {json.loads(l)["id"] for l in open(LOG)}

rng=np.random.default_rng(500)                       # new seed
new_mix = rng.dirichlet([ALPHA]*len(DOMAINS), size=N_NEW)
done = done_ids()
print(f"{N_NEW} broad proxies (alpha={ALPHA}), ids {START_ID}-{START_ID+N_NEW-1}")
for j in range(N_NEW):
    jid = START_ID + j
    if jid in done:
        print(f"  skip {jid} (done)"); continue
    m = new_mix[j]
    t0=time.time()
    vl = M.train_one_proxy(m.tolist(), log_every=10**9)
    rec={"id":jid,"mixture":m.tolist(),"vloss":vl,
         "target":target_metric(vl),"sec":round(time.time()-t0,1),"phase":"sweep2"}
    with open(LOG,"a") as f: f.write(json.dumps(rec)+"\n")
    print(f"[sweep2 {j+1}/{N_NEW}] id={jid} target={rec['target']:.4f} "
          f"mix={[round(x,3) for x in m.tolist()]} ({rec['sec']}s)", flush=True)
print("done.")

Writing /content/regmix/src/sweep_more.py


In [43]:
# 1. log intact at 53?
!wc -l /content/regmix/runs/sweep_log.jsonl
# 2. GPU free? (want ~40 GB — restart if a zombie survived from the diagnostic cells)
import torch; print(f"free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
# 3. backup thread still alive? if unsure, restart it (the threading.Thread backup_loop cell)

53 /content/regmix/runs/sweep_log.jsonl
free: 41.1 GB


In [45]:
import json, numpy as np, lightgbm as lgb
recs=[json.loads(l) for l in open("/content/regmix/runs/sweep_log.jsonl")]
DOMAINS=["web","code","maths","indic","papers"]
X=np.array([r["mixture"] for r in recs])
TW={"web":0.60,"code":0.10,"maths":0.10,"indic":0.15,"papers":0.05}
y=np.array([sum(TW[d]*r["vloss"][d] for d in DOMAINS) for r in recs])
g=lgb.train(dict(objective="regression",num_leaves=7,min_data_in_leaf=8,learning_rate=0.03,
    lambda_l1=0.5,lambda_l2=0.5,verbose=-1), lgb.Dataset(X,y), num_boost_round=500)
opt=[]
for s in range(20):
    rng=np.random.default_rng(s); n=100_000
    web=rng.uniform(0.55,0.65,size=n)
    rest=rng.dirichlet([1.0]*4,size=n)*(1-web)[:,None]
    cand=np.column_stack([web,rest])
    opt.append(cand[g.predict(cand).argmin()])
opt=np.array(opt)
print("web-constrained optimum (mean ± std over 20 seeds):")
for i,d in enumerate(DOMAINS): print(f"  {d:8s} {opt[:,i].mean():.3f} ± {opt[:,i].std():.3f}")

web-constrained optimum (mean ± std over 20 seeds):
  web      0.571 ± 0.011
  code     0.104 ± 0.015
  maths    0.114 ± 0.017
  indic    0.148 ± 0.012
  papers   0.064 ± 0.012
